# Initial population

Declarative base totals and splits replace hand-built `y0` arrays.
Shares are normalised; properties nobody mentioned split evenly over carriers
(ragged-aware). Evaluation runs once at run start.

Each section is a different way to say where the people start, and
each figure is that distribution. The seed is 10 infectious out of
1000. On the ragged map, `infect` exists only on `active`, so that
third of the population is halved again. The reachable-mass line
should be straight with slope 1000. `by=` and `where=` then break the
even default in the ways the bars show.


In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio
import jax
import jax.numpy as jnp

from summer4 import (
    REMAINDER,
    Everything,
    FlowModel,
    InitialPopulation,
    Param,
    Property,
    PropertyMap,
    Split,
    EntryFlow,
)

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

## 1. Total plus seed

Base entries are **totals** for the rows they match. `R` is unnamed,
so it stays empty; the figure should be 990 susceptible and 10
infectious.


In [ ]:
state = Property("state", ("S", "I", "R"))
pmap = PropertyMap.from_property(state)
plan = InitialPopulation(
    {state["S"]: Param("pop") - Param("seed"), state["I"]: Param("seed")}
).compile(pmap)
y = plan.evaluate({"pop": 1000.0, "seed": 10.0})
assert float(np.asarray(y.data)[pmap.select(state["S"])][0]) == 990.0
assert float(np.asarray(y.data)[pmap.select(state["I"])][0]) == 10.0
assert float(np.asarray(y.data).sum()) == 1000.0

mass = pd.Series(
    {
        name: float(np.asarray(y.data)[pmap.select(state[name])].sum())
        for name in state.traits
    }
)
mass.to_frame("people").plot.bar(
    title="Ten seed cases; the rest of the total is susceptible",
    labels={"index": "state", "value": "people"},
)


## 2. Even default on a ragged map

`infect` exists only under `active`. Unspecified properties split `1/K`
on carriers only. For one age band the bars should be equal naive and
incipient shares, with active halved into low and high.


In [ ]:
age = Property("age", ("0", "5", "15"))
state = Property("state", ("naive", "incipient", "active"))
infect = Property("infect", ("low", "high"))
pmap = (
    PropertyMap.from_property(age)
    .stratify(state)
    .stratify(infect, where=state["active"])
)
print(pmap.to_frame())
y = InitialPopulation({age[a]: 1000.0 for a in age.traits}).compile(pmap).evaluate({})
data = np.asarray(y.data)
for a in age.traits:
    assert np.isclose(float(data[pmap.select(age[a])].sum()), 1000.0)
    naive = float(data[pmap.select(age[a] & state["naive"])][0])
    low = float(data[pmap.select(age[a] & state["active"] & infect["low"])][0])
    assert np.isclose(naive, 1000.0 / 3.0)
    assert np.isclose(low, 1000.0 / 6.0)
    assert np.isclose(low, naive / 2.0)

band = age.traits[0]
pieces = {
    "naive": float(data[pmap.select(age[band] & state["naive"])].sum()),
    "incipient": float(data[pmap.select(age[band] & state["incipient"])].sum()),
    "active, low": float(data[pmap.select(age[band] & state["active"] & infect["low"])].sum()),
    "active, high": float(data[pmap.select(age[band] & state["active"] & infect["high"])].sum()),
}
pd.Series(pieces).to_frame(f"people in age {band}").plot.bar(
    title="Even split, and infect only where it exists",
    labels={"index": "compartment", "value": "people"},
)


## 3. Parameter split with `REMAINDER` and `jax.grad`

`reachable` takes `frac` of the susceptible total and `unreachable`
takes the rest. Reachable mass against `frac` should be a straight
line through the origin with slope 1000 — that slope is the gradient
the assert checks at `frac = 0.3`.


In [ ]:
reach = Property("reach", ("reachable", "unreachable"))
pmap = PropertyMap.from_property(state := Property("state", ("S", "I"))).stratify(reach)
plan = InitialPopulation(
    {state["S"]: Param("pop")},
    splits=(Split(reach, {"reachable": Param("frac"), "unreachable": REMAINDER}),),
).compile(pmap)

def reachable_mass(frac):
    y = plan.evaluate({"pop": 1000.0, "frac": frac})
    return jnp.sum(jnp.asarray(y.data)[pmap.select(reach["reachable"])])

g = float(jax.grad(reachable_mass)(0.3))
assert np.isclose(g, 1000.0, rtol=1e-5)

fracs = np.linspace(0.05, 0.95, 19)
pd.DataFrame(
    {"reachable": [float(reachable_mass(frac)) for frac in fracs]},
    index=fracs,
).plot(
    title="Reachable mass is frac x 1000",
    labels={"index": "frac", "value": "people"},
)


## 4. Split by an arbitrary function with `by=`

The callable returns a share of `imm=yes` for each age: 0.8 of the
young and 0.3 of the old, out of 100 people spread evenly across age.
The bars should read 40 and 15 immune.


In [ ]:
age = Property("age", ("young", "old"))
imm = Property("imm", ("yes", "no"))
pmap = PropertyMap.from_property(age).stratify(imm)
y = (
    InitialPopulation(
        {Everything(): 100.0},
        splits=(
            Split(
                imm,
                lambda p: jnp.array([[p["vy"], 1 - p["vy"]], [p["vo"], 1 - p["vo"]]]),
                by=(age,),
            ),
        ),
    )
    .compile(pmap)
    .evaluate({"vy": 0.8, "vo": 0.3})
)
data = np.asarray(y.data)
assert np.isclose(float(data[pmap.select(age["young"] & imm["yes"])][0]), 40.0)
assert np.isclose(float(data[pmap.select(age["old"] & imm["yes"])][0]), 15.0)

immune = pd.Series(
    {
        "young, immune": float(data[pmap.select(age["young"] & imm["yes"])][0]),
        "young, not": float(data[pmap.select(age["young"] & imm["no"])][0]),
        "old, immune": float(data[pmap.select(age["old"] & imm["yes"])][0]),
        "old, not": float(data[pmap.select(age["old"] & imm["no"])][0]),
    }
)
immune.to_frame("people").plot.bar(
    title="by= applies a different immune share in each age",
    labels={"index": "compartment", "value": "people"},
)


## 5. `where=` as summer2 `adjust_population_split`

Age splits 60/40. The vaccine split applies only to the old, so the
young stay even across doses and the old follow 70/20/10. Both age
bands are drawn.


In [ ]:
state = Property("state", ("S", "I"))
age = Property("age", ("young", "old"))
vacc = Property("vacc", ("one", "two", "none"))
pmap = PropertyMap.from_property(state).stratify(age).stratify(vacc)
y = (
    InitialPopulation(
        {state["S"]: 990.0},
        splits=(
            Split(age, {"young": 0.6, "old": 0.4}),
            Split(vacc, {"one": 0.7, "two": 0.2, "none": 0.1}, where=age["old"]),
        ),
    )
    .compile(pmap)
    .evaluate({})
)
data = np.asarray(y.data)
old = data[pmap.select(state["S"] & age["old"])]
np.testing.assert_allclose(old, 990.0 * 0.4 * np.array([0.7, 0.2, 0.1]))
young = data[pmap.select(state["S"] & age["young"])]
np.testing.assert_allclose(young, 990.0 * 0.6 / 3.0)

labels = [f"{a}, {v}" for a in age.traits for v in vacc.traits]
values = [
    float(data[pmap.select(state["S"] & age[a] & vacc[v])][0])
    for a in age.traits
    for v in vacc.traits
]
pd.Series(values, index=labels).to_frame("susceptibles").plot.bar(
    title="Vaccine split applies to the old only",
    labels={"index": "age, dose", "value": "people"},
)


## 6–7. `initial_state` and warm-start `y0` override

`set_initial_population` is what `initial_state` returns. A `y0`
passed to `run` replaces it for that solve — here a one-step run that
saves only t = 0, so the bar is the warm start, not the declared 7.


In [ ]:
state = Property("state", ("Y",))
pmap = PropertyMap.from_property(state)
model = FlowModel(pmap)
model.add_flow(EntryFlow("in", state["Y"], 0.0))
model.set_initial_population({state["Y"]: 7.0})
cm = model.compile()
assert float(np.asarray(cm.initial_state({}).data)[0]) == 7.0
from summer4 import PropertyData

warm = PropertyData.wrap(pmap, np.array([99.0]))
res = cm.run({}, warm, t0=0.0, dt=1.0, steps=0, solver="euler")
assert float(np.asarray(res["compartments"].values.data)[0, 0]) == 99.0

pd.Series(
    {
        "declared initial_state": float(np.asarray(cm.initial_state({}).data)[0]),
        "y0 passed to run": float(np.asarray(res["compartments"].values.data)[0, 0]),
    }
).to_frame("people").plot.bar(
    title="A y0 argument overrides the declared initial population",
    labels={"index": "source", "value": "people"},
)
